# 06 — DMRG Comparison (TeNPy)

> **spinq-vqe** | ARPA Quantum Logical Systems (QONDRA)

Classical DMRG reference energies for the 1D Kagome strip using [TeNPy](https://github.com/tenpy/tenpy).
Compares against exact diagonalization (where available) and VQE results from NB02/NB05.

**Install:** `pip install -e ".[dmrg]"`

## What this establishes

| Check | Purpose |
|-------|--------|
| Hamiltonian match | TeNPy MPO reproduces PennyLane ED matrix |
| N=9 validation | DMRG E₀ matches ED to < 0.01% |
| N=12, 18, 24 | Polynomial-scaling reference beyond ED reach |
| χ convergence (N=18) | Bond-dimension residual certificate (not available for VQE) |
| Entanglement profile | DMRG MPS vs VQE statevector in bits (NB03) |

In [ ]:
import csv
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from spinq_vqe import dmrg, entanglement

REPO = Path('..').resolve()
DATA = REPO / 'data'
FIG = REPO / 'figures'

N_CELLS = [3, 4, 6, 8]
# N=9 is exact at χ≈4; use N=18 where truncation is nontrivial.
CHI_SWEEP_N_CELLS = 6
CHI_SWEEP = [8, 16, 32, 64, 128, 200, 300, 400]
CHI_MAX = 400

---
## 1. Validate Hamiltonian against PennyLane ED

In [ ]:
max_diff = dmrg.validate_hamiltonian_against_pennylane(3)
print(f'max |H_tenpy - H_pennylane| at N=9: {max_diff:.3e}')

---
## 2. DMRG ground-state energies

In [ ]:
results = []
for n_cells in N_CELLS:
    res = dmrg.run_dmrg(n_cells, chi_max=CHI_MAX, compute_entropies=(n_cells == 3))
    results.append(res)
    print(
        f'N={res.n_sites:>2}  E0={res.e0:.8f}  chi={res.chi}  trunc={res.truncation_error:.2e}'
    )

csv_path = dmrg.save_dmrg_reference_csv(results, DATA / 'dmrg_reference_energies.csv')
print(f'Saved -> {csv_path.name}')

---
## 3. Comparison table: ED vs DMRG vs VQE

In [ ]:
def _load_csv_energy(path, key_n, key_e):
    out = {}
    with path.open(newline='', encoding='utf-8') as handle:
        for row in csv.DictReader(handle):
            out[int(row[key_n])] = float(row[key_e])
    return out

ed = _load_csv_energy(DATA / 'ed_reference_energies.csv', 'n_sites', 'E0_normalized')
dmrg_e = {r.n_sites: r.e0 for r in results}
vqe = {}
with (DATA / 'vqe_scaling.csv').open(newline='', encoding='utf-8') as handle:
    for row in csv.DictReader(handle):
        if row['E_VQE'] not in ('N/A', ''):
            vqe[int(row['N'])] = float(row['E_VQE'])

rows = []
for n in [9, 12, 18, 24]:
    e_ed = ed.get(n)
    e_dmrg = dmrg_e.get(n)
    e_vqe = vqe.get(n)
    err = None if e_vqe is None or e_dmrg is None else abs(e_vqe - e_dmrg) / abs(e_dmrg) * 100
    rows.append({
        'N': n,
        'ED E0': f'{e_ed:.4f}' if e_ed is not None else '—',
        'DMRG E0': f'{e_dmrg:.4f}' if e_dmrg is not None else '—',
        'VQE E0': f'{e_vqe:.4f}' if e_vqe is not None else '—',
        'VQE err vs DMRG (%)': f'{err:.2f}' if err is not None else '—',
    })
df = pd.DataFrame(rows)
display(df)

---
## 4. χ convergence (N=18)

N=9 is exact already at χ≈4, so a flat energy-vs-χ curve is not a useful certificate.
At N=18 the bond dimension truncates and the residual |E(χ)−E(χ_max)| drops with χ.

In [ ]:
n_chi = 3 * CHI_SWEEP_N_CELLS
chi_results = dmrg.run_dmrg_chi_sweep(CHI_SWEEP_N_CELLS, CHI_SWEEP, max_sweeps=60)
e_ref = chi_results[-1].e0
chis = [r.chi_max for r in chi_results]
residuals = [max(abs(r.e0 - e_ref), 1e-16) for r in chi_results]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
ax = axes[0]
ax.plot(chis, [r.e0 for r in chi_results], 'o-', color='#8B6FA8', lw=2, ms=6)
ax.axhline(e_ref, color='#C97B7B', ls='--', lw=1.2, label=f'E0(chi={CHI_MAX})')
ax.set_xlabel('Bond dimension chi_max')
ax.set_ylabel('E0 (normalized per site)')
ax.set_title(f'DMRG energy vs chi (N={n_chi})', fontweight='semibold', color='#333')
ax.set_xscale('log', base=2)
ax.legend(fontsize=9)

ax = axes[1]
ax.semilogy(chis, residuals, 'o-', color='#8B6FA8', lw=2, ms=6)
ax.set_xlabel('Bond dimension chi_max')
ax.set_ylabel('|E0(chi) - E0(chi_max)|')
ax.set_title(f'Convergence residual (N={n_chi})', fontweight='semibold', color='#333')
ax.set_xscale('log', base=2)
plt.tight_layout()
plt.savefig(FIG / 'dmrg_chi_convergence.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5. Entanglement: DMRG MPS vs VQE statevector

In [ ]:
# DMRG entropies are in bits (converted from TeNPy nats in dmrg.run_dmrg).
# VQE overlay uses NB03 half-chain convention (|A| <= N/2); full-chain VQE RDMs
# are numerically unreliable for this approximate statevector.
n9 = next(r for r in results if r.n_sites == 9)
cuts = list(range(1, n9.n_sites))
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(cuts, n9.entropies, 'o-', color='#8B6FA8', lw=2, ms=5, label='DMRG MPS')
sv_path = DATA / 'statevector_hea_best.npy'
if sv_path.exists():
    sv = np.load(sv_path)
    profile = entanglement.entanglement_profile(sv, n9.n_sites)
    ax.plot(
        profile['subsystem_sizes'],
        profile['entropies'],
        's--',
        color='#7EB8D4',
        lw=2,
        ms=5,
        label='VQE (NB03, |A|<=N/2)',
    )
else:
    print('Note: run NB02 first for VQE statevector overlay.')
ax.set_xlabel('Bond cut index')
ax.set_ylabel('Entanglement entropy (bits)')
ax.set_title('Bipartite entropy profile (N=9)', fontweight='semibold', color='#333')
ax.legend()
plt.tight_layout()
plt.savefig(FIG / 'dmrg_entanglement_profile.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 6. ED / DMRG / VQE comparison figure

In [ ]:
sizes = sorted(set(ed) | set(vqe) | set(dmrg_e))
x = np.arange(len(sizes))
width = 0.25
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(x - width, [ed.get(n, np.nan) for n in sizes], width, label='ED', color='#EBD8DC')
ax.bar(x, [dmrg_e.get(n, np.nan) for n in sizes], width, label='DMRG', color='#C97B7B')
ax.bar(x + width, [vqe.get(n, np.nan) for n in sizes], width, label='VQE best', color='#7EB8D4')
ax.set_xticks(x, [str(n) for n in sizes])
ax.set_xlabel('System size N (sites)')
ax.set_ylabel('E0 (normalized per site)')
ax.set_title('ED vs DMRG vs VQE', fontweight='semibold', color='#333')
ax.legend()
plt.tight_layout()
plt.savefig(FIG / 'dmrg_vqe_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Regenerate all NB06 figures via: python scripts/run_dmrg_benchmark.py')